# LLM Finetuning & Inference with Predibase

This notebook handles finetuning and inference for open-source models using Predibase:
- **Models**: Llama-2 (7B, 13B, 70B), Llama-3 (8B, 70B), Mistral-7B, Mixtral-8x7B, Gemma (2B, 7B), CodeLlama, Zephyr, Phi-3, DeepSeek
- **Tasks**: Bandgap (regression), Dielectric (regression), CrystalStructure (classification), MatKG (link prediction)

## Workflow
1. Upload training data to Predibase
2. Finetune models with LoRA adapters
3. Run inference (base and finetuned) with 10 iterations
4. Extract numerical predictions using GPT-3.5
5. Save results for analysis

## 1. Setup & Configuration

In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
import random
import time
import asyncio
import re
from tqdm import tqdm
from tqdm.asyncio import tqdm as atqdm
from concurrent.futures import ThreadPoolExecutor

from predibase import Predibase, FinetuningConfig
from openai import OpenAI

# API Keys - Replace with your own or load from environment
PREDIBASE_API_TOKEN = os.getenv('PREDIBASE_API_TOKEN', 'your_predibase_token')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your_openai_key')

# Initialize clients
pb = Predibase(api_token=PREDIBASE_API_TOKEN)
oclient = OpenAI(api_key=OPENAI_API_KEY)

print(f"Predibase connected")

In [ ]:
# Configuration
DATA_DIR = '../data'
RESULTS_DIR = '../results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Tasks and their properties
TASKS = {
    'Bandgap': {'type': 'regression', 'unit': 'eV'},
    'Dielectric': {'type': 'regression', 'unit': ''},
    'CrystalStructure': {'type': 'classification', 'classes': ['Trigonal', 'Cubic', 'Tetragonal', 'Hexagonal', 'Monoclinic', 'Orthorhombic', 'Triclinic']},
    'MatKG': {'type': 'link_prediction'}
}

# Models available on Predibase
MODELS = {
    'llama-2-7b-chat': 'llama-2-7b-chat',
    'llama-2-13b-chat': 'llama-2-13b-chat',
    'llama-2-70b-chat': 'llama-2-70b-chat',
    'llama-3-8b-instruct': 'llama-3-8b-instruct',
    'llama-3-70b-instruct': 'llama-3-70b-instruct',
    'mistral-7b-instruct-v0-2': 'mistral-7b-instruct-v0-2',
    'mistral-7b-instruct-v0-3': 'mistral-7b-instruct-v0-3',
    'mixtral-8x7b-instruct-v0-1': 'mixtral-8x7b-instruct-v0-1',
    'gemma-2b-instruct': 'gemma-2b-instruct',
    'gemma-7b-instruct': 'gemma-7b-instruct',
    'codellama-7b-instruct': 'codellama-7b-instruct',
    'codellama-13b-instruct': 'codellama-13b-instruct',
    'codellama-70b-instruct': 'codellama-70b-instruct',
    'zephyr-7b-beta': 'zephyr-7b-beta',
    'phi-3-mini-4k-instruct': 'phi-3-mini-4k-instruct',
    'deepseek-r1-distill-qwen-32b': 'deepseek-r1-distill-qwen-32b'
}

# Number of inference iterations
N_ITERATIONS = 10

## 2. Data Loading Functions

In [ ]:
def load_data(task, model, split='Train'):
    """
    Load training or test data for a task-model combination.
    
    Data format (batched):
    - prompt: The prompt template
    - completion: List of ground truth values
    - Formula: List of chemical formulas
    - mpid: List of Materials Project IDs
    """
    filepath = os.path.join(DATA_DIR, task, f'{task}_{model}_{split}.csv')
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return None
    return pd.read_csv(filepath)

def parse_list_column(val):
    """Parse a string representation of a list into an actual list."""
    import ast
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except:
            return [val]
    return [val]

# Test loading
sample_df = load_data('Bandgap', 'llama-2-7b-chat', 'Train')
if sample_df is not None:
    print(f"Loaded {len(sample_df)} batches")
    print(f"Columns: {sample_df.columns.tolist()}")

## 3. Finetuning with Predibase

In [ ]:
def upload_dataset_to_predibase(task, model):
    """
    Upload training dataset to Predibase.
    Returns the dataset name for use in finetuning.
    """
    filepath = os.path.join(DATA_DIR, task, f'{task}_{model}_Train.csv')
    dataset_name = f"{task}_{model}_train"
    
    try:
        # Upload to Predibase
        dataset = pb.datasets.from_file(filepath, name=dataset_name)
        print(f"Uploaded dataset: {dataset_name}")
        return dataset_name
    except Exception as e:
        print(f"Error uploading dataset: {e}")
        return None

def finetune_model(base_model, dataset_name, task, epochs=3):
    """
    Finetune a model on Predibase using LoRA.
    
    Args:
        base_model: Name of the base model on Predibase
        dataset_name: Name of the uploaded dataset
        task: Task name for adapter naming
        epochs: Number of training epochs
    
    Returns:
        Adapter name if successful, None otherwise
    """
    adapter_name = f"{base_model}-{task}-finetuning"
    
    config = FinetuningConfig(
        base_model=base_model,
        epochs=epochs,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # LoRA target modules
        learning_rate=2e-4,
        rank=16  # LoRA rank
    )
    
    try:
        # Start finetuning job
        job = pb.finetuning.jobs.create(
            config=config,
            dataset=dataset_name,
            repo=adapter_name
        )
        print(f"Started finetuning job: {job.id}")
        print(f"Adapter will be saved as: {adapter_name}")
        return adapter_name, job.id
    except Exception as e:
        print(f"Error starting finetuning: {e}")
        return None, None

def check_finetuning_status(job_id):
    """Check the status of a finetuning job."""
    job = pb.finetuning.jobs.get(job_id)
    return job.status

In [ ]:
# Example: Finetune a single model
# Uncomment to run

# TASK = 'Bandgap'
# MODEL = 'llama-2-7b-chat'
# 
# # Upload dataset
# dataset_name = upload_dataset_to_predibase(TASK, MODEL)
# 
# # Start finetuning
# if dataset_name:
#     adapter_name, job_id = finetune_model(MODEL, dataset_name, TASK)
#     print(f"Finetuning started. Job ID: {job_id}")

## 4. Inference Functions

In [ ]:
async def run_inference_batch(prompts, base_model, adapter=None, max_tokens=256):
    """
    Run batch inference on Predibase.
    
    Args:
        prompts: List of prompts to process
        base_model: Base model name
        adapter: Adapter name (None for base model inference)
        max_tokens: Maximum tokens to generate
    
    Returns:
        List of generated completions
    """
    try:
        if adapter:
            lorax_client = pb.deployments.client("pb://deployments/default")
            responses = await lorax_client.generate_batch(
                prompts=prompts,
                adapter_id=adapter,
                max_new_tokens=max_tokens
            )
        else:
            lorax_client = pb.deployments.client(f"pb://deployments/{base_model}")
            responses = await lorax_client.generate_batch(
                prompts=prompts,
                max_new_tokens=max_tokens
            )
        return [r.generated_text for r in responses]
    except Exception as e:
        print(f"Inference error: {e}")
        return [None] * len(prompts)

def extract_numerical_value(text, task):
    """
    Extract numerical prediction from LLM output using GPT-3.5.
    
    Args:
        text: Raw LLM output
        task: Task type ('Bandgap', 'Dielectric', 'CrystalStructure')
    
    Returns:
        Extracted value (float for regression, string for classification)
    """
    if task in ['Bandgap', 'Dielectric']:
        prompt = f"""Extract the numerical value from this text. Return ONLY the number, nothing else.
If multiple numbers are present, return the one that represents the {task.lower()} prediction.
If no valid number is found, return 'None'.

Text: {text}

Number:"""
    else:  # CrystalStructure
        prompt = f"""Extract the crystal structure from this text. Return ONLY one of these options:
Trigonal, Cubic, Tetragonal, Hexagonal, Monoclinic, Orthorhombic, Triclinic

Text: {text}

Crystal Structure:"""
    
    try:
        response = oclient.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=20
        )
        result = response.choices[0].message.content.strip()
        
        if task in ['Bandgap', 'Dielectric']:
            try:
                return float(result)
            except:
                return None
        else:
            return result if result in TASKS['CrystalStructure']['classes'] else None
    except Exception as e:
        return None

## 5. Run Full Experiment

In [ ]:
async def run_experiment(task, model, adapter=None, n_iterations=10):
    """
    Run full inference experiment for a task-model combination.
    
    Args:
        task: Task name
        model: Model name
        adapter: Adapter name (None for base inference)
        n_iterations: Number of inference iterations
    
    Returns:
        DataFrame with results
    """
    # Load test data
    test_df = load_data(task, model, 'Test')
    if test_df is None:
        return None
    
    results = []
    
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc=f"{task}-{model}"):
        prompt = row['prompt']
        completions = parse_list_column(row['completion'])
        formulas = parse_list_column(row['Formula'])
        mpids = parse_list_column(row.get('mpid', [''] * len(completions)))
        
        # Run multiple iterations
        for iteration in range(n_iterations):
            # Run inference
            outputs = await run_inference_batch([prompt], model, adapter)
            raw_output = outputs[0] if outputs else None
            
            # Extract predictions for each sample in batch
            # Note: Need to handle batch output parsing
            if raw_output:
                # Parse batch output (format depends on prompt template)
                try:
                    preds = parse_list_column(raw_output)
                except:
                    preds = [None] * len(completions)
            else:
                preds = [None] * len(completions)
            
            # Record results
            for i, (formula, mpid, ground, pred) in enumerate(zip(formulas, mpids, completions, preds)):
                results.append({
                    'mpid': mpid,
                    'Formula': formula,
                    'Ground': ground,
                    f'prediction_{iteration+1}': pred,
                    'iteration': iteration + 1
                })
    
    return pd.DataFrame(results)

In [ ]:
def compute_summary_statistics(results_df, task):
    """
    Compute summary statistics from raw results.
    
    - Mode prediction (most common across iterations)
    - Prediction entropy (consistency measure)
    """
    from collections import Counter
    from scipy.stats import entropy
    
    pred_cols = [c for c in results_df.columns if c.startswith('prediction_')]
    
    summary_records = []
    
    for mpid, group in results_df.groupby('mpid'):
        record = {
            'mpid': mpid,
            'Formula': group['Formula'].iloc[0],
            'Ground': group['Ground'].iloc[0]
        }
        
        # Collect all predictions
        all_preds = []
        for col in pred_cols:
            vals = group[col].dropna().tolist()
            all_preds.extend(vals)
            record[col] = vals[0] if vals else None
        
        # Mode prediction
        if all_preds:
            if task in ['Bandgap', 'Dielectric']:
                # For regression: use median
                valid_preds = [p for p in all_preds if p is not None]
                record['mode_prediction'] = np.median(valid_preds) if valid_preds else None
            else:
                # For classification: use mode
                counter = Counter(all_preds)
                record['mode_prediction'] = counter.most_common(1)[0][0] if counter else None
            
            # Entropy (discretize for regression)
            if task in ['Bandgap', 'Dielectric']:
                # Bin predictions for entropy calculation
                bins = np.histogram_bin_edges(valid_preds, bins='auto') if valid_preds else [0]
                hist, _ = np.histogram(valid_preds, bins=bins)
                record['prediction_entropy'] = entropy(hist + 1e-10)  # Add small value to avoid log(0)
            else:
                counts = list(Counter(all_preds).values())
                record['prediction_entropy'] = entropy(counts) if counts else 0
        
        summary_records.append(record)
    
    return pd.DataFrame(summary_records)

## 6. Example: Run Single Experiment

In [ ]:
# Example: Run base inference for a single model-task
# Uncomment to run

# TASK = 'Bandgap'
# MODEL = 'llama-3-8b-instruct'
# 
# # Base inference
# results = await run_experiment(TASK, MODEL, adapter=None, n_iterations=10)
# 
# # Compute summary
# summary = compute_summary_statistics(results, TASK)
# 
# # Save
# summary.to_csv(f'{RESULTS_DIR}/{TASK}_{MODEL}_ft_base_summary.csv', index=False)
# print(f"Saved: {TASK}_{MODEL}_ft_base_summary.csv")

## 7. Batch Run: All Models × All Tasks

In [ ]:
async def run_all_experiments():
    """
    Run experiments for all model-task combinations.
    Includes: base inference, same-task FT, and cross-task FT.
    """
    finetune_conditions = ['base'] + list(TASKS.keys())
    
    for task in TASKS.keys():
        for model in MODELS.keys():
            for ft_condition in finetune_conditions:
                output_file = f'{RESULTS_DIR}/{task}_{model}_ft_{ft_condition}_summary.csv'
                
                # Skip if already done
                if os.path.exists(output_file):
                    print(f"Skipping (exists): {output_file}")
                    continue
                
                # Determine adapter
                adapter = None if ft_condition == 'base' else f"{model}-{ft_condition}-finetuning/1"
                
                print(f"\nRunning: {task} | {model} | FT: {ft_condition}")
                
                try:
                    results = await run_experiment(task, model, adapter=adapter, n_iterations=N_ITERATIONS)
                    
                    if results is not None and len(results) > 0:
                        summary = compute_summary_statistics(results, task)
                        summary.to_csv(output_file, index=False)
                        print(f"Saved: {output_file}")
                except Exception as e:
                    print(f"Error: {e}")

# Uncomment to run all experiments
# await run_all_experiments()

## 8. Download Trained Adapters

After finetuning is complete, download adapters for local use or HuggingFace upload.

In [ ]:
import zipfile

def download_adapter(adapter_name, output_dir='./adapters'):
    """
    Download a trained adapter from Predibase.
    
    Args:
        adapter_name: Full adapter name with version (e.g., 'llama-2-7b-chat-Bandgap-finetuning/1')
        output_dir: Directory to save the adapter
    """
    os.makedirs(output_dir, exist_ok=True)
    
    name, version = adapter_name.split('/')
    zip_filename = f"{name}-{version}.zip"
    zip_path = os.path.join(output_dir, zip_filename)
    
    try:
        print(f"Downloading {adapter_name}...")
        pb.adapters.download(adapter_name, dest=zip_path)
        print(f"  ✓ Saved to {zip_path}")
        
        # Unzip
        extract_path = os.path.join(output_dir, f"{name}-{version}")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print(f"  ✓ Extracted to {extract_path}")
        
        return extract_path
    except Exception as e:
        print(f"  ✗ Error: {e}")
        return None

# Example: Download all adapters for a task
# for model in MODELS.keys():
#     adapter_name = f"{model}-Bandgap-finetuning/1"
#     download_adapter(adapter_name)

## Notes

### File Naming Convention
- Input: `{task}_{model}_{split}.csv` (e.g., `Bandgap_llama-2-7b-chat_Train.csv`)
- Output: `{task}_{model}_ft_{finetune_condition}_summary.csv`

### Adapters on HuggingFace
Trained adapters are available at: `vinven7/{model}-{task}-finetuning-{version}`

### Cross-Task Evaluation
- `ft_base`: Base model without finetuning
- `ft_Bandgap`: Model finetuned on Bandgap task
- `ft_Dielectric`: Model finetuned on Dielectric task
- etc.

This allows studying transfer learning effects across tasks.